<div class="blog-language-switch" role="group" aria-label="Article language"><span aria-current="page">English</span><a href="/ipynb/zh-CN/Computer-Science/Computer-Organization/03-combinational-logic-and-alu.html" lang="zh-CN" hreflang="zh-CN">中文</a></div>

[Back to Computer Organization and Architecture guideline](Computer-Organization.html)


## **Combinational Logic and the ALU** {#combinational-logic-and-the-alu}

Chapter 02 established how a fixed-width bit pattern can represent an integer and how arithmetic should be interpreted. This chapter asks the next hardware question: **what physical organization transforms input bits into the required output bits?** The answer begins with a Boolean function and ends with a network of logic gates.

A **combinational circuit** is a circuit whose settled output depends only on its current inputs. It has no stored history. If the same input pattern is presented twice, the same output pattern must eventually appear both times. An adder, multiplexer, decoder, comparator, and arithmetic logic unit (ALU) are combinational. A register or counter is not, because its output also depends on previously stored state; those circuits belong to Chapter 04.

The word *eventually* matters. Gates are physical devices, so a changed input takes a small amount of time to propagate through them. Combinational describes the mathematical dependency, not instantaneous behavior. Correct design therefore has two obligations:

1. **functional correctness:** after signals settle, every input combination produces the specified output;
2. **timing correctness:** the longest path settles before another component samples the output.

::: {.diagram-scroll}
![A combinational component is refined from a requirement into a truth table, Boolean expression, gate network, reusable block, and finally part of an ALU.](assets/logic-design-flow.svg){fig-align="center" width="100%"}
:::

The recurring example is an 8-bit ALU asked to add 120 and 20. The binary addition produces `10001100`, which is 140 as an unsigned result but -116 under an 8-bit two's-complement interpretation. The bits alone cannot report whether either interpretation exceeded its range. The ALU therefore also computes status signals: unsigned carry is 0, while signed overflow is 1. By the end of the chapter, every stage from one-bit gates to that complete result will be explicit.

The chapter follows the normal direction of hardware construction:

`digital signals -> truth tables -> Boolean algebra -> gates -> reusable combinational blocks -> adders -> ALU -> area-delay tradeoffs`


### **From Boolean Values to Digital Signals** {#from-boolean-values-to-digital-signals}

Boolean algebra uses two abstract values, 0 and 1. A real circuit instead carries voltages, currents, and charges that vary continuously. Digital hardware works because it defines **ranges** of physical values rather than demanding one exact voltage. A low range is interpreted as logic 0, a high range as logic 1, and the region between them is not a valid steady input.

For illustration, a technology might guarantee that an input at or below 0.8 V is low and an input at or above 2.0 V is high. Those numbers are only an example; actual thresholds depend on the logic family and supply voltage. The unused interval provides room for noise and transition. A gate should produce output levels comfortably inside the valid ranges accepted by the next gate. The difference between a guaranteed output level and the corresponding input threshold is a **noise margin**.

| Electrical idea | Boolean abstraction | Why it matters |
|---|---|---|
| low voltage range | 0 | tolerates small disturbances without changing meaning |
| high voltage range | 1 | gives a second stable, distinguishable state |
| transition/undefined region | neither guaranteed 0 nor 1 | signals should pass through it, not remain there |
| output drive and fan-out | one output feeding several inputs | excessive load increases delay and may violate levels |
| finite rise and fall time | a Boolean change after delay | prevents treating gates as instantaneous equations |

At the logic level, a signal is written as a Boolean variable such as $A$, $B$, or $S$. A combinational block with input vector $X$ and output vector $Y$ implements

$$
Y(t+t_{pd}) = F(X(t)).
$$

- $X(t)$ is the input pattern presented at time $t$.
- $F$ is the Boolean function implemented by the circuit.
- $t_{pd}$ is a conservative propagation delay through the block.
- $Y(t+t_{pd})$ is the output guaranteed to match the function after that delay.

This expression does not claim that the output is wrong for the entire interval. Some paths may settle sooner. It says a designer should not rely on the final answer before the worst-case delay has passed.

A logic diagram normally omits transistor-level details and treats each gate as an ideal Boolean operator with an attached delay and drive capability. This abstraction is powerful because the same Boolean design can be implemented in different technologies. Transistor sizing, voltage, fabrication, and wire capacitance change cost and timing, while the truth table remains the functional contract.


### **Logic Gates and Truth Tables** {#logic-gates-and-truth-tables}

A **logic gate** is a small circuit that implements a Boolean operator. A bubble on a gate symbol denotes inversion. NOT has one input; the common AND, OR, XOR, NAND, NOR, and XNOR gates shown below have two inputs but can be generalized to more.

![Standard symbols for AND, OR, and NOT gates.](assets/logic-gates-symbols.svg){fig-align="center" width="86%"}

*Image source: [LogicGates.svg](https://commons.wikimedia.org/wiki/File:LogicGates.svg), Vaughan Pratt, CC BY 3.0.*

A **truth table** lists the output for every possible input pattern. Two Boolean inputs have $2^2=4$ patterns. The table is therefore a complete specification, not a collection of representative examples.

| $A$ | $B$ | NOT $A$ | $A$ AND $B$ | $A$ OR $B$ | $A$ XOR $B$ | $A$ NAND $B$ | $A$ NOR $B$ | $A$ XNOR $B$ |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 0 | 0 | 1 | 0 | 0 | 0 | 1 | 1 | 1 |
| 0 | 1 | 1 | 0 | 1 | 1 | 1 | 0 | 0 |
| 1 | 0 | 0 | 0 | 1 | 1 | 1 | 0 | 0 |
| 1 | 1 | 0 | 1 | 1 | 0 | 0 | 0 | 1 |

AND expresses **all required conditions**: its output is 1 only when both inputs are 1. OR expresses **at least one condition**. XOR is 1 when the inputs differ; for several inputs it represents odd parity. XNOR is 1 when two inputs agree, which makes it a natural building block for equality comparison. NAND and NOR invert AND and OR respectively.

The gate names describe functions, not necessarily one physical transistor arrangement. A synthesis tool may replace an AND followed by a NOT with a NAND gate, factor shared logic, or use technology-specific compound gates, provided the truth table is preserved.

NAND and NOR are called **universal gates** because either gate type alone can express every Boolean function. For NAND, written with the operator $\uparrow$,

$$
\neg A=A\uparrow A,
$$

$$
A\land B=(A\uparrow B)\uparrow(A\uparrow B),
$$

$$
A\lor B=(A\uparrow A)\uparrow(B\uparrow B).
$$

The first equation ties both NAND inputs together, so the output is NOT $A$. The second inverts the NAND result to recover AND. The third applies De Morgan's law: invert both inputs and NAND them to recover OR. Once NOT, AND, and OR are available, any sum-of-products expression can be constructed. Universality is useful for manufacturing libraries and formal reasoning, although a practical chip normally offers many gate types to reduce area and delay.

<details>
<summary>Python experiment: generate gate truth tables and verify NAND universality</summary>

~~~python
from itertools import product


def as_bit(value: bool | int) -> int:
    """Normalize a Boolean-like value to the hardware symbols 0 or 1."""
    return 1 if bool(value) else 0


def nand(a: int, b: int) -> int:
    return as_bit(not (a and b))


def gate_row(a: int, b: int) -> dict[str, int]:
    return {
        "A": a,
        "B": b,
        "NOT A": as_bit(not a),
        "AND": as_bit(a and b),
        "OR": as_bit(a or b),
        "XOR": a ^ b,
        "NAND": nand(a, b),
        "NOR": as_bit(not (a or b)),
        "XNOR": as_bit(a == b),
    }


rows = [gate_row(a, b) for a, b in product((0, 1), repeat=2)]
for row in rows:
    print(row)

# Construct NOT, AND, and OR using NAND only.
for a, b in product((0, 1), repeat=2):
    not_a_from_nand = nand(a, a)
    and_from_nand = nand(nand(a, b), nand(a, b))
    or_from_nand = nand(nand(a, a), nand(b, b))

    assert not_a_from_nand == as_bit(not a)
    assert and_from_nand == as_bit(a and b)
    assert or_from_nand == as_bit(a or b)
~~~

</details>


### **Boolean Algebra and Logic Simplification** {#boolean-algebra-and-logic-simplification}

Boolean algebra manipulates expressions whose variables are only 0 or 1. The familiar symbols are

- $\neg A$, $\overline{A}$, or `!A` for NOT;
- $A\land B$, $AB$, or `A & B` for AND;
- $A\lor B$, $A+B$, or `A | B` for OR;
- $A\oplus B$ or `A ^ B` for XOR.

The `+` in Boolean algebra means OR, not integer addition. In particular, $1+1=1$ as a Boolean expression. Context and notation must be stated clearly when arithmetic and logic appear in the same ALU.

Different expressions may implement the same truth table but require different hardware. Consider

$$
F=AB+A\overline{B}+\overline{A}B.
$$

The first two terms factor as $A(B+\overline{B})=A$. The remaining expression is $A+\overline{A}B$, and absorption reduces it to

$$
F=A+B.
$$

The original direct implementation needs three AND terms, two inversions, and an OR network. The simplified expression needs one OR gate. Both are functionally identical, but the second normally has less area, lower capacitance, fewer opportunities for glitches, and a shorter propagation path.

The identities used most often in circuit simplification are:

| Identity | Boolean form | Intuition |
|---|---|---|
| identity | $A+0=A$, $A\cdot1=A$ | combining with the neutral value changes nothing |
| domination | $A+1=1$, $A\cdot0=0$ | one controlling value determines the result |
| idempotence | $A+A=A$, $AA=A$ | repeating the same condition adds no information |
| complement | $A+\overline A=1$, $A\overline A=0$ | a bit is either 0 or 1, never both |
| absorption | $A+AB=A$, $A(A+B)=A$ | the broader condition already includes the narrower one |
| distributive | $A(B+C)=AB+AC$ | supports factoring and expansion |
| De Morgan | $\overline{AB}=\overline A+\overline B$ | negating “both” means at least one input is false |
| De Morgan | $\overline{A+B}=\overline A\,\overline B$ | negating “either” means both inputs are false |

De Morgan's laws are especially important when bubbles move through a schematic. Inverting an AND output is equivalent to ORing inverted inputs; inverting an OR output is equivalent to ANDing inverted inputs. This allows a designer or synthesis tool to map a function efficiently into NAND, NOR, or compound CMOS gates.

Algebraic simplification is not merely cosmetic. It changes physical structure. However, the expression with the fewest written literals is not automatically the best implementation. Sharing a term can reduce area but increase fan-out; adding redundant logic can remove a hazard; a regular structure can route better than an irregular minimum. Functional equivalence is the first requirement, followed by technology-aware optimization.

<details>
<summary>Python verification: prove two Boolean expressions equivalent by exhaustion</summary>

~~~python
from itertools import product


def original_expression(a: int, b: int) -> int:
    # F = AB + A(not B) + (not A)B
    return int((a and b) or (a and not b) or (not a and b))


def simplified_expression(a: int, b: int) -> int:
    # F = A + B
    return int(a or b)


def equivalent(function_a, function_b, input_count: int) -> bool:
    """Check all 2^input_count combinations and report a counterexample."""
    for inputs in product((0, 1), repeat=input_count):
        left = function_a(*inputs)
        right = function_b(*inputs)
        if left != right:
            print("counterexample:", inputs, left, right)
            return False
    return True


assert equivalent(original_expression, simplified_expression, 2)
print("Equivalent for all four input combinations.")
~~~

</details>


#### **Canonical Forms** {#canonical-forms}

A truth table can always be translated mechanically into a Boolean expression. **Canonical forms** make that translation systematic by requiring every term to mention every input variable.

A **minterm** is an AND term that is 1 for exactly one input row. For variables $A$, $B$, and $C$, the row `101` produces the minterm

$$
m_5=A\overline{B}C.
$$

The subscript 5 is the unsigned value of the input pattern `101`. A variable appears uncomplemented when its row value is 1 and complemented when its row value is 0. ORing the minterms for every row where the required output is 1 gives the canonical **sum of products** (SOP).

For the three-input majority function, the output is 1 whenever at least two inputs are 1. Its 1-rows are 3, 5, 6, and 7:

$$
M(A,B,C)=\Sigma m(3,5,6,7)
=\overline A BC+A\overline B C+AB\overline C+ABC.
$$

- $\Sigma m$ means “OR these minterms.”
- Each product is 1 on exactly one selected row.
- The canonical expression is guaranteed correct, but it is not yet economical.

Factoring or a Karnaugh map reduces the majority function to

$$
M=AB+AC+BC.
$$

Each remaining product says one pair is simultaneously 1. If any two inputs are 1, at least one product is 1.

A **maxterm** is an OR term that is 0 for exactly one row. ANDing maxterms for all output-0 rows gives the canonical **product of sums** (POS). The majority function has zero rows 0, 1, 2, and 4:

$$
M(A,B,C)=\Pi M(0,1,2,4)
$$

$$
=(A+B+C)(A+B+\overline C)(A+\overline B+C)(\overline A+B+C).
$$

In a maxterm, a variable is uncomplemented when the row contains 0 because an OR term is zero only when all its literals are zero. The notation therefore reverses the minterm rule.

| Form | Built from | Selects truth-table rows | Natural two-level implementation |
|---|---|---|---|
| canonical SOP | minterms joined by OR | rows where output is 1 | AND plane followed by OR |
| canonical POS | maxterms joined by AND | rows where output is 0 | OR plane followed by AND |

Canonical forms prove that gates are expressive enough to implement any finite Boolean function. Their drawback is size: an $n$-input truth table has $2^n$ rows, and a direct canonical circuit can grow exponentially. Simplification and structured building blocks are therefore essential.


#### **Karnaugh Maps** {#karnaugh-maps}

A **Karnaugh map** (K-map) rearranges a truth table so neighboring cells differ in exactly one input bit. It uses Gray-code order such as `00, 01, 11, 10`, not ordinary binary order. When two adjacent 1-cells are combined, the variable that changes between them disappears from the product term.

![A four-variable Karnaugh map uses overlapping power-of-two groups to remove changing variables.](assets/karnaugh-map.svg){fig-align="center" width="48%"}

*Image source: [Karnaugh.svg](https://commons.wikimedia.org/wiki/File:Karnaugh.svg), Mobius, public domain.*

The simplification procedure is:

1. copy each truth-table output into its Gray-code cell;
2. group adjacent 1s in rectangles containing $1,2,4,8,\ldots$ cells;
3. make groups as large as possible, because every doubling removes one literal;
4. allow groups to overlap when that creates larger terms;
5. treat opposite edges as adjacent, so a group may wrap around the map;
6. translate each group into the variables that remain constant inside it;
7. OR the resulting product terms.

For example,

$$
F(A,B,C,D)=\Sigma m(0,1,2,3,8,9,10,11,13,15)
$$

contains an eight-cell group where $B=0$, giving $\overline B$. A four-cell group covers minterms 9, 11, 13, and 15, where $A=1$ and $D=1$, giving $AD$. The simplified result is

$$
F=\overline B+AD.
$$

$C$ disappears because it changes within both groups. $A$ and $D$ also change inside the eight-cell group. The map makes this elimination visible rather than relying on a long algebraic derivation.

An input combination marked **don't care**, commonly written `X`, may be treated as either 0 or 1 when simplifying. This is valid only when the combination cannot occur or its output is irrelevant under the interface contract. A don't-care cell may enlarge a useful group, but it does not have to be included.

K-maps are excellent for roughly two to six variables and for teaching adjacency. Larger functions are handled by algorithmic minimizers and synthesis tools because a $2^n$ visual grid becomes unmanageable. Even then, the underlying principle remains: combine cases that differ in irrelevant variables.


### **Combinational Circuit Design** {#combinational-circuit-design}

Combinational design begins with behavior, not with gates. A reliable workflow is:

1. **Define the interface.** Name every input and output, state active-high or active-low conventions, and declare invalid input combinations.
2. **Specify behavior.** Use a truth table, equations, or a precise algorithm. Multi-bit blocks are often clearer as arithmetic or selection rules than as enormous truth tables.
3. **Derive Boolean functions.** Produce one function for each output bit.
4. **Simplify and structure.** Factor common terms or replace repeated patterns with multiplexers, decoders, adders, and shifters.
5. **Map to a gate library.** Account for available gates, fan-in limits, fan-out, and wiring.
6. **Verify function.** Compare every legal input against the specification or prove sub-block contracts compositionally.
7. **Analyze timing and cost.** Find the critical path, estimate area and power, and revise if constraints are missed.

The design should be acyclic at the combinational level. Feeding an output back into an input without a state element creates a combinational loop. Such a loop may oscillate, settle unpredictably, or depend on analog delays. Intentional feedback belongs inside characterized storage elements such as latches and flip-flops.

As a small example, suppose three replicated sensors vote on one control decision. The output must be 1 when at least two sensors report 1. The behavioral phrase “at least two” becomes the majority function $AB+AC+BC$. Three two-input AND gates and an OR network implement it directly. The same block can be verified independently and then reused without reconsidering every internal gate.

#### **Functional Correctness** {#functional-correctness}

Functional correctness means the implemented circuit and specification produce the same settled output for every permitted input. For $n$ Boolean inputs, exhaustive simulation requires $2^n$ cases. This is practical for a small gate network but not for a 64-bit ALU, whose input space is astronomically large.

Larger circuits are verified through several complementary methods:

- **compositional reasoning:** prove each sub-block satisfies a contract, then prove the connections preserve those contracts;
- **directed tests:** exercise boundaries such as all zeroes, all ones, sign changes, and maximum carry chains;
- **random and constrained-random tests:** cover combinations a human may not anticipate;
- **assertions and property checking:** state invariants such as `sum == (a + b) mod 2^n`;
- **formal equivalence:** mathematically compare an optimized implementation with a trusted reference.

For a combinational block, a complete functional specification should also define illegal or unknown inputs. Hardware-description languages use values such as `X` for unknown and `Z` for high impedance during simulation. Treating every `X` as either 0 or 1 can hide initialization and contention problems, while allowing uncontrolled `X` propagation can make debugging noisy. The interface should determine which behavior is meaningful.

<details>
<summary>Python verification: specify, implement, and exhaustively test a majority voter</summary>

~~~python
from itertools import product


def majority_specification(a: int, b: int, c: int) -> int:
    """Behavioral rule: at least two of the three inputs are one."""
    return int(a + b + c >= 2)


def majority_gate_network(a: int, b: int, c: int) -> int:
    """Structural Boolean implementation: AB + AC + BC."""
    ab = a & b
    ac = a & c
    bc = b & c
    return ab | ac | bc


for inputs in product((0, 1), repeat=3):
    expected = majority_specification(*inputs)
    observed = majority_gate_network(*inputs)
    assert observed == expected, (inputs, expected, observed)
    print(inputs, "->", observed)
~~~

</details>

#### **Propagation Delay** {#propagation-delay}

When a gate input changes, the output does not switch immediately. Two common delay measurements are

$$
t_{PLH}: \text{output low-to-high propagation delay},
$$

$$
t_{PHL}: \text{output high-to-low propagation delay}.
$$

A conservative single delay is often written

$$
t_{pd}=\max(t_{PLH},t_{PHL}).
$$

The **critical path** is the input-to-output path with the greatest total delay. If a path passes through gates with delays $d_1,d_2,\ldots,d_k$, its estimated delay is

$$
t_{path}=\sum_{i=1}^{k} d_i+t_{wire}.
$$

- $k$ is the number of logic stages on that path.
- $d_i$ is the propagation delay of stage $i$ under its load.
- $t_{wire}$ covers interconnect delay, which can dominate in large modern circuits.
- The block delay is the maximum $t_{path}$ over all input-to-output paths.

::: {.diagram-scroll}
![A red three-gate critical path determines when the output is guaranteed valid; a shorter path settles sooner but does not set the worst case.](assets/critical-path-timing.svg){fig-align="center" width="100%"}
:::

Signals can take paths of different length and briefly produce a wrong intermediate output called a **glitch** or **hazard**. A later register normally samples only after the circuit has settled, so a short internal glitch may be harmless functionally, but it still consumes dynamic power and can be dangerous when driving asynchronous control or clock signals.

Propagation delay differs from **contamination delay**, the earliest time after an input change that any output might begin changing. If one register launches data to another, the maximum propagation delay helps determine the minimum clock period, while minimum contamination delay participates in hold-time analysis. Chapter 04 will connect these limits to clocks and registers.

<details>
<summary>Python model: calculate arrival times and identify a critical path</summary>

~~~python
from dataclasses import dataclass


@dataclass(frozen=True)
class Gate:
    inputs: tuple[str, ...]
    delay_ns: float


def analyze_arrival_times(
    primary_inputs: set[str],
    gates_in_topological_order: dict[str, Gate],
) -> tuple[dict[str, float], dict[str, list[str]]]:
    """Compute latest arrival time and one critical predecessor chain."""
    arrival = {name: 0.0 for name in primary_inputs}
    path = {name: [name] for name in primary_inputs}

    for output, gate in gates_in_topological_order.items():
        # The latest input controls when this gate can produce a valid output.
        latest_input = max(gate.inputs, key=lambda name: arrival[name])
        arrival[output] = arrival[latest_input] + gate.delay_ns
        path[output] = path[latest_input] + [output]

    return arrival, path


gates = {
    "and_ab": Gate(("A", "B"), 0.8),
    "xor_c": Gate(("and_ab", "C"), 1.1),
    "Y": Gate(("xor_c", "D"), 0.7),
}

arrival, path = analyze_arrival_times({"A", "B", "C", "D"}, gates)
assert abs(arrival["Y"] - 2.6) < 1e-12
assert path["Y"] in (["A", "and_ab", "xor_c", "Y"],
                     ["B", "and_ab", "xor_c", "Y"])

print("output arrival:", arrival["Y"], "ns")
print("critical path:", " -> ".join(path["Y"]))
~~~

</details>


### **Multiplexers, Decoders, and Encoders** {#multiplexers-decoders-and-encoders}

A processor repeatedly needs to select one value, activate one destination, or translate between compact and one-hot control. Multiplexers, decoders, and encoders package those patterns into reusable combinational blocks.

A **multiplexer** (MUX) selects one of several data inputs and forwards it to one output. A 2-to-1 MUX has data inputs $D_0$ and $D_1$, select input $S$, and output

$$
Y=\overline S D_0+SD_1.
$$

When $S=0$, the first product becomes $D_0$ and the second is forced to 0. When $S=1$, the roles reverse. The select signal controls *which path matters* without modifying the selected data.

![A 2-to-1 multiplexer can be implemented by gating each data input with the appropriate select literal and ORing the paths.](assets/multiplexer-logic.svg){fig-align="center" width="64%"}

*Image source: [Multiplexer2.svg](https://commons.wikimedia.org/wiki/File:Multiplexer2.svg), CaesarIII, CC BY-SA 3.0.*

An $n$-bit select field can choose among as many as $2^n$ inputs. Wide datapath multiplexers repeat the one-bit selection circuit for every bit while sharing the select lines. An ALU uses a MUX to select among arithmetic, logic, shift, and comparison results. A processor datapath uses MUXes to choose register operands, immediate values, next program-counter values, and write-back data.

A MUX can also implement any Boolean function. Use some variables as select bits and connect each data input to 0, 1, another variable, or its complement according to the function's truth table. This observation links arbitrary logic to structured selection hardware.

A **decoder** maps an $n$-bit binary code to up to $2^n$ one-hot outputs. For a 2-to-4 decoder with enable $E$,

$$
Y_0=E\overline A\overline B,\quad
Y_1=E\overline A B,\quad
Y_2=EA\overline B,\quad
Y_3=EAB.
$$

Exactly one $Y_i$ is 1 when enabled. Each output is a minterm, so a decoder generates all canonical input cases at once.

![A 2-to-4 decoder shows the relationship among input code, minterms, gate network, and one-hot outputs.](assets/decoder-2-to-4.svg){fig-align="center" width="78%"}

*Image source: [Decoder Example.svg](https://commons.wikimedia.org/wiki/File:Decoder_Example.svg), BlueJester0101, CC BY-SA 3.0; the local copy fixes the labels to the English fallback.*

Decoders select registers, memory banks, I/O devices, instruction classes, and individual control actions. A **demultiplexer** is related but routes one data input to a selected output; a decoder instead asserts a selected control line.

An **encoder** performs the opposite coding direction: a one-hot input becomes a compact binary index. A simple encoder assumes at most one input is active. A **priority encoder** defines which index wins when several inputs are 1, often choosing the highest-numbered request. It also emits a `valid` bit so all-zero input is distinguishable from “input zero is active.” Interrupt controllers use priority encoders to choose among simultaneous requests.

| Block | Main inputs | Main output | Core question | Processor example |
|---|---|---|---|---|
| multiplexer | many data values + select | one selected value | which value should pass? | choose ALU result or next PC |
| decoder | compact code + enable | one-hot lines | which destination is named? | select a register or control action |
| encoder | one-hot requests | compact code | which source is active? | encode an interrupt request |
| priority encoder | possibly multiple requests | winning code + valid | which active source has priority? | select highest-priority exception |

<details>
<summary>Python implementation: multiplexer, decoder, and priority encoder</summary>

~~~python
def mux(inputs: list[int], select: int) -> int:
    """Select one input; the input count must be a power of two."""
    if not inputs or len(inputs) & (len(inputs) - 1):
        raise ValueError("multiplexer input count must be a power of two")
    if not 0 <= select < len(inputs):
        raise ValueError("select code is out of range")
    return inputs[select]


def decoder(code: int, input_width: int, enable: bool = True) -> list[int]:
    """Return 2^input_width one-hot outputs."""
    output_count = 1 << input_width
    if not 0 <= code < output_count:
        raise ValueError("code does not fit the decoder width")
    outputs = [0] * output_count
    if enable:
        outputs[code] = 1
    return outputs


def priority_encoder(requests: list[int]) -> tuple[int, bool]:
    """Choose the highest-numbered asserted request."""
    for index in range(len(requests) - 1, -1, -1):
        if requests[index]:
            return index, True
    return 0, False


assert mux([10, 20, 30, 40], 2) == 30
assert decoder(2, input_width=2) == [0, 0, 1, 0]
assert decoder(2, input_width=2, enable=False) == [0, 0, 0, 0]
assert priority_encoder([0, 1, 0, 1]) == (3, True)
assert priority_encoder([0, 0, 0, 0]) == (0, False)
~~~

</details>


### **Adder Circuits** {#adder-circuits}

An $n$-bit addition is built by solving one binary column and repeating that solution across bit positions. At position $i$, the circuit receives operand bits $a_i$ and $b_i$ plus carry-in $c_i$. It emits sum bit $s_i$ and carry-out $c_{i+1}$. The carry connects the local one-bit decision to the next more significant position.

The arithmetic total of the three one-bit inputs is 0, 1, 2, or 3. The low bit of that total becomes $s_i$, and the high bit becomes $c_{i+1}$:

$$
a_i+b_i+c_i=2c_{i+1}+s_i.
$$

This is an ordinary integer equation. $s_i$ is the remainder modulo 2, while $c_{i+1}$ records whether the column total contains one group of two.

#### **Half and Full Adders** {#half-and-full-adders}

A **half adder** combines two bits but has no carry-in:

$$
S=A\oplus B,
$$

$$
C=AB.
$$

XOR is the low bit of $A+B$, and AND detects the only case that produces a carry. A half adder is suitable for the least-significant column only when its incoming carry is known to be 0.

| $A$ | $B$ | half-adder $S$ | half-adder $C$ |
|---:|---:|---:|---:|
| 0 | 0 | 0 | 0 |
| 0 | 1 | 1 | 0 |
| 1 | 0 | 1 | 0 |
| 1 | 1 | 0 | 1 |

A **full adder** includes $C_{in}$ and is therefore reusable at every position:

$$
S=A\oplus B\oplus C_{in},
$$

$$
C_{out}=AB+C_{in}(A\oplus B).
$$

The sum is 1 when an odd number of the three inputs is 1. The carry is 1 when $A$ and $B$ generate a carry directly, or when exactly one of them is 1 and the incoming carry propagates through.

![A full adder can be constructed from two half adders followed by an OR gate for their carry outputs.](assets/full-adder-modules.svg){fig-align="center" width="72%"}

*Image source: [Full Adder Modules.svg](https://commons.wikimedia.org/wiki/File:Full_Adder_Modules.svg), Inductiveload, public domain.*

The diagram exposes a useful hierarchy. The first half adder combines $A$ and $B$. The second combines its partial sum with $C_{in}$. At most one of the two half-adders can generate a carry in any input case, so OR joins their carry outputs.

<details>
<summary>Python implementation: derive half-adder and full-adder truth tables</summary>

~~~python
from itertools import product


def half_adder(a: int, b: int) -> tuple[int, int]:
    sum_bit = a ^ b
    carry = a & b
    return sum_bit, carry


def full_adder(a: int, b: int, carry_in: int) -> tuple[int, int]:
    # First half adder combines the operand bits.
    partial_sum, carry_ab = half_adder(a, b)

    # Second half adder incorporates the carry from the previous column.
    sum_bit, carry_partial = half_adder(partial_sum, carry_in)

    # Either half adder may produce the outgoing carry.
    carry_out = carry_ab | carry_partial
    return sum_bit, carry_out


for inputs in product((0, 1), repeat=3):
    sum_bit, carry_out = full_adder(*inputs)
    arithmetic_total = sum(inputs)
    assert arithmetic_total == 2 * carry_out + sum_bit
    print(inputs, "-> sum", sum_bit, "carry", carry_out)
~~~

</details>

#### **Ripple-Carry Addition** {#ripple-carry-addition}

A **ripple-carry adder** connects $n$ full adders in a chain. The carry-out of bit $i$ becomes the carry-in of bit $i+1$. The structure is regular, compact, and easy to scale.

![A four-bit ripple-carry adder chains four full adders from the least-significant carry input to the final carry output.](assets/ripple-carry-adder.svg){fig-align="center" width="82%"}

*Image source: [4-bit ripple carry adder.svg](https://commons.wikimedia.org/wiki/File:4-bit_ripple_carry_adder.svg), Cburnett, CC BY-SA 3.0.*

For each bit,

$$
s_i=a_i\oplus b_i\oplus c_i,
$$

$$
c_{i+1}=a_ib_i+c_i(a_i\oplus b_i).
$$

The result is functionally available only after every required carry has propagated. In the worst case, such as `1111 + 0001`, the low bit changes the carry into bit 1, which changes the carry into bit 2, and so on. The stored four-bit result becomes `0000`, and the final carry-out is 1.

For an $n$-bit adder with approximate per-stage carry delay $t_c$ and final sum delay $t_s$, a simple worst-case model is

$$
t_{RCA}\approx(n-1)t_c+t_s.
$$

- $n-1$ carry transitions may be required before the most-significant stage knows its input carry.
- $t_c$ is the carry-in to carry-out delay of one full-adder stage.
- $t_s$ is the final carry-in to sum delay.
- The linear dependence on $n$ is called **linear logic depth**.

The ripple design often wins for small widths or area-sensitive circuits because it uses little extra logic and wiring. At wide datapath widths or high clock rates, carry delay becomes a bottleneck.

<details>
<summary>Python implementation: ripple addition with a per-bit carry trace</summary>

~~~python
def ripple_add(a: int, b: int, width: int, carry_in: int = 0) -> tuple[int, int, list[dict[str, int]]]:
    """Add fixed-width patterns using one full-adder step per bit."""
    mask = (1 << width) - 1
    if not 0 <= a <= mask or not 0 <= b <= mask:
        raise ValueError("operands must fit the declared width")

    result = 0
    carry = carry_in
    trace: list[dict[str, int]] = []

    for bit_index in range(width):
        a_bit = (a >> bit_index) & 1
        b_bit = (b >> bit_index) & 1
        carry_before = carry
        sum_bit, carry = full_adder(a_bit, b_bit, carry_before)
        result |= sum_bit << bit_index

        trace.append({
            "bit": bit_index,
            "a": a_bit,
            "b": b_bit,
            "carry_in": carry_before,
            "sum": sum_bit,
            "carry_out": carry,
        })

    return result, carry, trace


result, carry_out, trace = ripple_add(0b1111, 0b0001, width=4)
assert result == 0b0000
assert carry_out == 1

for step in trace:
    print(step)
~~~

</details>

#### **Carry-Lookahead Intuition** {#carry-lookahead-intuition}

Carry lookahead reduces waiting by asking each bit position two questions that do not depend on its incoming carry:

$$
P_i=a_i\oplus b_i \qquad \text{(propagate)},
$$

$$
G_i=a_ib_i \qquad \text{(generate)}.
$$

$G_i=1$ means the bit produces a carry regardless of $c_i$. $P_i=1$ means exactly one operand bit is 1, so an incoming carry will pass through. The recurrence becomes

$$
c_{i+1}=G_i+P_ic_i.
$$

Substituting earlier carries expands the dependency. For the first two positions,

$$
c_1=G_0+P_0c_0,
$$

$$
c_2=G_1+P_1G_0+P_1P_0c_0.
$$

The three terms for $c_2$ mean: bit 1 generates; or bit 1 propagates a carry generated by bit 0; or both bits propagate the external carry $c_0$. Similar expressions compute $c_3$ and $c_4$ directly from the $P_i$, $G_i$, and $c_0$ signals instead of waiting for a physical ripple.

::: {.diagram-scroll}
![Each bit computes propagate and generate signals in parallel; a shared lookahead network then produces all carries for the final XOR sums.](assets/carry-lookahead-flow.svg){fig-align="center" width="100%"}
:::

The full expansion grows rapidly in gate fan-in and wiring. Practical wide adders use hierarchy: small groups compute group-propagate and group-generate signals, and a second level looks ahead across groups. Tree adders and prefix adders organize the same information with roughly logarithmic depth.

| Design | Carry logic depth | Extra logic and wiring | Typical strength |
|---|---|---|---|
| ripple carry | grows approximately as $O(n)$ | low | compact and efficient at modest widths |
| flat lookahead | small for a short group | high fan-in grows quickly | fast for 4- or 8-bit groups |
| hierarchical/tree lookahead | approximately $O(\log n)$ | larger, more wiring | high-speed wide datapaths |

Big-O notation here describes how logic depth scales, not software running time. Every adder still exists physically and evaluates concurrently. The engineering question is how many dependent gate levels lie on the worst path.

<details>
<summary>Python model: compute four-bit lookahead carries in parallel form</summary>

~~~python
def carry_lookahead_4(a: int, b: int, carry_in: int = 0) -> tuple[int, int]:
    if not 0 <= a < 16 or not 0 <= b < 16:
        raise ValueError("this model accepts four-bit operands")

    a_bits = [(a >> i) & 1 for i in range(4)]
    b_bits = [(b >> i) & 1 for i in range(4)]

    # Local facts are independent and can be formed simultaneously in hardware.
    p = [a_bits[i] ^ b_bits[i] for i in range(4)]
    g = [a_bits[i] & b_bits[i] for i in range(4)]
    c0 = carry_in

    # These equations are written without a sequential carry recurrence.
    c1 = g[0] | (p[0] & c0)
    c2 = g[1] | (p[1] & g[0]) | (p[1] & p[0] & c0)
    c3 = (g[2] | (p[2] & g[1]) | (p[2] & p[1] & g[0])
          | (p[2] & p[1] & p[0] & c0))
    c4 = (g[3] | (p[3] & g[2]) | (p[3] & p[2] & g[1])
          | (p[3] & p[2] & p[1] & g[0])
          | (p[3] & p[2] & p[1] & p[0] & c0))

    carries = [c0, c1, c2, c3]
    sum_bits = [p[i] ^ carries[i] for i in range(4)]
    result = sum(bit << i for i, bit in enumerate(sum_bits))
    return result, c4


for a in range(16):
    for b in range(16):
        result, carry = carry_lookahead_4(a, b)
        assert result == (a + b) & 0xF
        assert carry == int(a + b >= 16)

print(carry_lookahead_4(0b1111, 0b0001))
~~~

</details>


### **Comparators and Shifters** {#comparators-and-shifters}

A **comparator** reports relationships such as equal, less than, or greater than. Equality is naturally bit-parallel. Two $n$-bit words are equal only when every corresponding pair agrees:

$$
EQ=\bigwedge_{i=0}^{n-1}(a_i\operatorname{\ XNOR\ }b_i).
$$

$\bigwedge$ means AND over all positions. Each XNOR emits 1 for a matching pair, and the final AND requires every pair to match.

Unsigned magnitude comparison is lexicographic from the most-significant bit. The first position where $a_i\ne b_i$ decides the result: $a_i=1,b_i=0$ means $A>B$. Lower bits cannot overturn a more significant difference because one unit at position $i$ outweighs all lower positions combined.

Signed two's-complement comparison needs care. If sign bits differ, the negative operand is smaller. If signs agree, the remaining comparison can be performed consistently, or the ALU can subtract and combine the sign of the result with signed overflow. A negative wrapped subtraction result is not by itself reliable when overflow occurs.

A **shifter** moves bit positions. For an $n$-bit word:

- logical left shift inserts zeroes on the right and discards high bits;
- logical right shift inserts zeroes on the left;
- arithmetic right shift repeats the sign bit to preserve two's-complement sign;
- rotate moves discarded bits around to the opposite end.

A simple one-position shifter is mostly wiring. A **barrel shifter** chooses any shift amount in one combinational operation. For an 8-bit word, three MUX stages conditionally shift by 1, 2, and 4 positions according to the three bits of the shift amount. Any value from 0 through 7 is the sum of selected powers of two.

::: {.diagram-scroll}
![Magnitude comparison stops at the first differing high bit, while a barrel shifter composes selectable shifts by powers of two.](assets/comparator-shifter.svg){fig-align="center" width="100%"}
:::

An $n$-bit logarithmic barrel shifter uses about $\log_2n$ stages, each containing $n$ multiplexers. Its logic depth is $O(\log n)$ and its selection hardware is $O(n\log n)$. The alternative of shifting one position per clock uses less combinational area but takes multiple cycles and requires state. Which design is better depends on latency, throughput, and area targets.

Shifts are used for multiplication or division by powers of two, field extraction, address calculation, normalization, cryptography, and instruction encoding. Logical right shift implements unsigned division by powers of two. Arithmetic right shift often resembles signed division but rounding rules for negative values must match the ISA or language definition.

<details>
<summary>Python implementation: fixed-width comparison, shifts, and rotation</summary>

~~~python
def mask_for(width: int) -> int:
    if width <= 0:
        raise ValueError("width must be positive")
    return (1 << width) - 1


def decode_signed(pattern: int, width: int) -> int:
    sign_bit = 1 << (width - 1)
    pattern &= mask_for(width)
    return pattern - (1 << width) if pattern & sign_bit else pattern


def compare_words(a: int, b: int, width: int, *, signed: bool = False) -> int:
    """Return -1, 0, or 1 under the selected interpretation."""
    mask = mask_for(width)
    a &= mask
    b &= mask
    if signed:
        a = decode_signed(a, width)
        b = decode_signed(b, width)
    return (a > b) - (a < b)


def shift_or_rotate(value: int, amount: int, width: int, operation: str) -> int:
    mask = mask_for(width)
    value &= mask
    amount %= width

    if operation == "lsl":
        return (value << amount) & mask
    if operation == "lsr":
        return value >> amount
    if operation == "asr":
        return (decode_signed(value, width) >> amount) & mask
    if operation == "ror":
        return ((value >> amount) | (value << (width - amount))) & mask
    raise ValueError("operation must be lsl, lsr, asr, or ror")


assert compare_words(0b1010, 0b1001, 4) == 1
assert compare_words(0b1010, 0b1001, 4, signed=True) == 1
assert shift_or_rotate(0b1011_0110, 5, 8, "lsr") == 0b0000_0101
assert shift_or_rotate(0b1011_0110, 3, 8, "asr") == 0b1111_0110
assert shift_or_rotate(0b1011_0110, 4, 8, "ror") == 0b0110_1011

print(decode_signed(0b1010, 4), decode_signed(0b1001, 4))
~~~

</details>


### **Constructing an Arithmetic Logic Unit** {#constructing-an-arithmetic-logic-unit}

An **arithmetic logic unit** combines several combinational operations behind one interface. It accepts operand words $A$ and $B$, receives an operation code from control logic, produces result word $R$, and emits status information. The ALU does not decide which instruction is being executed and does not remember previous results. The instruction decoder supplies control; registers provide operands and store selected results.

![A symbolic ALU receives two integer operands and an opcode, then emits one integer result and status signals.](assets/alu-block.svg){fig-align="center" width="72%"}

*Image source: [ALU block.svg](https://commons.wikimedia.org/wiki/File:ALU_block.svg), D. Ilyin after Jim Lamberson, CC0 1.0.*

A straightforward organization computes several candidate outputs in parallel:

- bitwise AND, OR, and XOR apply one gate independently at every bit position;
- one adder performs addition and subtraction;
- a shifter provides logical and arithmetic shifts;
- comparison logic produces a Boolean result such as set-less-than;
- an output multiplexer selects the candidate named by the ALU control code.

Parallel candidates reduce operation-selection delay because the MUX chooses an already computed value, but inactive units still occupy area and may switch internally. A more area-constrained design can share hardware or take multiple cycles. The architectural operation is the same; the organization differs.

A compact illustrative control table is:

| Control | Operation | Result meaning |
|---:|---|---|
| `0000` | AND | $A\land B$ bit by bit |
| `0001` | OR | $A\lor B$ bit by bit |
| `0010` | XOR | $A\oplus B$ bit by bit |
| `0011` | ADD | $(A+B)\bmod2^n$ |
| `0100` | SUB | $(A+\overline B+1)\bmod2^n$ |
| `0101` | logical left shift | $A\ll(B\bmod n)$ |
| `0110` | logical right shift | unsigned $A\gg(B\bmod n)$ |
| `0111` | arithmetic right shift | signed $A\gg(B\bmod n)$ |
| `1000` | signed less-than | 1 if signed $A<B$, otherwise 0 |

Subtraction reuses the adder by conditionally inverting every $B$ bit and setting the low carry-in to 1:

$$
A-B=A+(\overline B+1).
$$

A bank of XOR gates can perform the conditional inversion because $b_i\oplus0=b_i$ for addition and $b_i\oplus1=\overline{b_i}$ for subtraction. The same add/subtract control bit can become $c_0$. This small structural choice avoids a separate subtractor.

The common flags for an $n$-bit result $R$ are:

$$
Z=1\quad\text{iff}\quad R=0,
$$

$$
N=R_{n-1},
$$

$$
C=c_n,
$$

where $Z$ is zero, $N$ copies the most-significant result bit, and $C$ is the adder's final carry-out. $N$ says only that the result pattern has a top bit of 1; it represents negativity only under two's-complement interpretation.

Signed overflow for addition is

$$
V_{add}=\overline{(a_{n-1}\oplus b_{n-1})}(a_{n-1}\oplus r_{n-1}).
$$

- $a_{n-1}$ and $b_{n-1}$ are the operand sign bits.
- $r_{n-1}$ is the result sign bit.
- The first factor requires equal operand signs.
- The second requires the result sign to differ from the first operand.

For subtraction,

$$
V_{sub}=(a_{n-1}\oplus b_{n-1})(a_{n-1}\oplus r_{n-1}).
$$

Subtraction overflows when the operands have different signs and the result sign unexpectedly differs from $A$. Carry and overflow answer different questions: carry describes unsigned range; overflow describes signed two's-complement range.

Return to the 8-bit example:

$$
120+20=140\quad\longrightarrow\quad R=10001100_2.
$$

The result is nonzero, so $Z=0$. Its top bit is 1, so $N=1$. The unsigned sum 140 fits in 0 through 255, so $C=0$. Two positive signed operands produced a negative-looking result, so $V=1$. The same output pattern is therefore valid unsigned arithmetic and overflowed signed arithmetic.

Flag conventions are part of the ISA. Some architectures expose all four flags, some expose fewer, and some define subtraction carry as “no borrow” while others describe a borrow separately. Never transfer a flag interpretation between architectures without checking the contract.

<details>
<summary>Python implementation: a fixed-width ALU with result and status flags</summary>

~~~python
from dataclasses import dataclass


@dataclass(frozen=True)
class ALUResult:
    value: int
    zero: int
    negative: int
    carry: int
    overflow: int


def signed_value(pattern: int, width: int) -> int:
    sign = 1 << (width - 1)
    pattern &= (1 << width) - 1
    return pattern - (1 << width) if pattern & sign else pattern


def alu(a: int, b: int, operation: str, width: int = 8) -> ALUResult:
    """Model an ALU; carry and overflow are meaningful for ADD/SUB."""
    if width <= 0:
        raise ValueError("width must be positive")

    mask = (1 << width) - 1
    sign_mask = 1 << (width - 1)
    a &= mask
    b &= mask
    carry = 0
    overflow = 0

    if operation == "AND":
        result = a & b
    elif operation == "OR":
        result = a | b
    elif operation == "XOR":
        result = a ^ b
    elif operation == "ADD":
        full = a + b
        result = full & mask
        carry = int(full > mask)
        same_operand_sign = not bool((a ^ b) & sign_mask)
        changed_result_sign = bool((a ^ result) & sign_mask)
        overflow = int(same_operand_sign and changed_result_sign)
    elif operation == "SUB":
        # Hardware form: A + NOT(B) + 1. Carry=1 conventionally means no borrow.
        full = a + ((~b) & mask) + 1
        result = full & mask
        carry = int(full > mask)
        different_operand_sign = bool((a ^ b) & sign_mask)
        changed_result_sign = bool((a ^ result) & sign_mask)
        overflow = int(different_operand_sign and changed_result_sign)
    elif operation == "LSL":
        result = (a << (b % width)) & mask
    elif operation == "LSR":
        result = a >> (b % width)
    elif operation == "ASR":
        result = (signed_value(a, width) >> (b % width)) & mask
    elif operation == "SLT":
        result = int(signed_value(a, width) < signed_value(b, width))
    else:
        raise ValueError(f"unsupported ALU operation: {operation}")

    return ALUResult(
        value=result,
        zero=int(result == 0),
        negative=int(bool(result & sign_mask)),
        carry=carry,
        overflow=overflow,
    )


example = alu(120, 20, "ADD", width=8)
assert example == ALUResult(0b1000_1100, 0, 1, 0, 1)

subtraction = alu(7, 10, "SUB", width=8)
assert subtraction.value == 0b1111_1101  # -3 in two's complement
assert subtraction.negative == 1
assert subtraction.carry == 0            # a borrow was required
assert subtraction.overflow == 0

assert alu(0xF0, 0x0F, "AND").value == 0
assert alu(0x80, 1, "ASR").value == 0xC0
assert alu(0xFF, 1, "SLT").value == 1    # signed -1 < +1

print(example)
print(subtraction)
~~~

</details>


### **Area, Delay, and Circuit Complexity** {#area-delay-and-circuit-complexity}

Two circuits can implement the same truth table yet differ substantially as engineering designs. The main dimensions are:

- **area:** transistors, standard cells, wiring, and physical layout occupied;
- **delay:** the worst input-to-output settling time, dominated by the critical path;
- **power and energy:** switching activity, capacitance, voltage, leakage, and frequency;
- **fan-out and wiring:** how many loads a signal drives and how far it travels;
- **regularity and verification cost:** how easy the design is to lay out, scale, and prove correct.

A first-order dynamic power model is

$$
P_{dynamic}\approx\alpha C V^2 f.
$$

- $\alpha$ is the average switching activity: the fraction of nodes changing per cycle.
- $C$ is the effective capacitance being charged and discharged.
- $V$ is supply voltage; its square makes voltage reduction especially powerful.
- $f$ is switching or clock frequency.

Simplifying logic can reduce $C$ by removing gates and wires. However, a faster architecture may add parallel hardware, increasing capacitance and switching. A glitch also contributes to $\alpha$ even if it disappears before the result is sampled.

Circuit complexity uses ideas related to algorithmic complexity but measures different resources. Software Big-O usually counts sequential operations as input size grows. A combinational circuit performs many gate evaluations concurrently; its important asymptotic measures are **size** (roughly gate count) and **depth** (longest dependent gate chain).

| Design choice | Area tendency | Delay tendency | Other consequence |
|---|---|---|---|
| canonical SOP | can be exponential in input count | potentially high fan-in | mechanically derived but rarely final |
| factored Boolean logic | often smaller | may add dependent levels | shared terms may increase fan-out |
| ripple-carry adder | small and regular | depth grows linearly with width | excellent for modest speed targets |
| lookahead/prefix adder | larger and wire-heavy | depth grows roughly logarithmically | suited to fast wide datapaths |
| one-bit iterative shifter | very small combinational block | multiple cycles for large shifts | requires state and control |
| barrel shifter | $O(n\log n)$ selection hardware | $O(\log n)$ stage depth | any shift completes in one combinational pass |
| parallel ALU candidates | more units switch or occupy area | fast final selection | simple control and high throughput |
| shared multi-cycle unit | less duplicated hardware | extra cycles and control | useful in area- or energy-constrained designs |

There is no universally best point. A tiny microcontroller, a high-frequency desktop CPU, and a low-power accelerator can implement the same arithmetic contract differently. Optimization begins with constraints: required width, operations, latency, clock period, throughput, power budget, and physical technology.

<details>
<summary>Python model: compare simple depth and hardware proxies for alternative designs</summary>

~~~python
import math


def design_proxies(width: int) -> dict[str, dict[str, int]]:
    """Return intentionally simple scaling proxies, not transistor estimates."""
    if width <= 1 or width & (width - 1):
        raise ValueError("use a power-of-two width greater than one")

    return {
        "ripple_adder": {
            "logic_depth": width,
            "relative_cells": width,
        },
        "tree_lookahead_adder": {
            "logic_depth": math.ceil(math.log2(width)) + 2,
            "relative_cells": width * math.ceil(math.log2(width)),
        },
        "barrel_shifter": {
            "logic_depth": math.ceil(math.log2(width)),
            "relative_muxes": width * math.ceil(math.log2(width)),
        },
    }


for width in (8, 16, 32, 64):
    print(width, design_proxies(width))
~~~

</details>

These numbers express scaling trends only. Real timing depends on gate fan-in, cell choice, load, placement, wire length, buffering, voltage, and process variation. Physical synthesis may choose a hybrid design that does not fit one textbook label.

**Chapter summary.** Digital logic turns voltage ranges into Boolean values and uses truth tables as complete functional contracts. Boolean algebra, canonical forms, and Karnaugh maps transform those contracts into smaller gate networks. Multiplexers select data, decoders create one-hot control, and encoders compress requests. Half and full adders solve one binary column; ripple carry favors regularity and area, while carry lookahead trades extra logic and wiring for shorter depth. Comparators inspect significance, and barrel shifters compose power-of-two selection stages. An ALU places these functions behind one control interface and reports zero, negative, carry, and overflow according to the operation and ISA convention. Every implementation must finally balance functional correctness with critical-path delay, area, switching power, and physical wiring. Chapter 04 adds clocks and storage so these stateless transformations can become a machine that advances through time.
